# 🗂️ Notebook 2: Distributed Lock Manager — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Designing the API

We'll walk through **bad → better → best** versions of the API so you can feel
*why* each field exists.

### 🚫 Bad v0 — "just a boolean"

```http
POST /locks/{name}/acquire
POST /locks/{name}/release
```

Problems:
1. No owner identity → anyone can release anyone's lock.
2. No TTL → a crashed client holds the lock forever.
3. No fencing token → a paused client can come back and corrupt data.
4. No renewal → long jobs must pick a huge TTL (bad liveness) or short (bad safety).

### ✅ Better v1 — add owner + TTL

```http
POST /locks/{name}/acquire   { owner, ttl_ms }
POST /locks/{name}/release   { owner }
```

Fixes #1 and #2. Still vulnerable to GC pauses (the split-brain scenario).

### 🏆 Best v2 — add fencing token + renewal

```http
POST /locks/{name}/acquire   { owner, ttl_ms }
  → { acquired: true, fencing_token: 17, expires_at: ... }

POST /locks/{name}/renew     { owner, token, ttl_ms }
  → { expires_at: ... }

POST /locks/{name}/release   { owner, token }
```

The **fencing token never goes down** — even across restarts of the lock service —
so whatever backs it must be a monotonic counter
(etcd's revision, Redis `INCR`, SQL sequence).


## Data model

| Field        | Value                                   |
|--------------|-----------------------------------------|
| `name`       | lock name (string key)                  |
| `owner`      | client id (UUID or `host:pid`)          |
| `token`      | monotonically increasing integer        |
| `expires_at` | timestamp (for lease-based eviction)    |

Let's pin this down with Pydantic so our types are checked at runtime.


In [ ]:
# Pydantic models for the v2 API. Running this cell proves our contracts validate.
from pydantic import BaseModel, Field, PositiveInt

class AcquireRequest(BaseModel):
    owner: str = Field(min_length=1, description="Client id, e.g. 'worker-7@host-a'")
    ttl_ms: PositiveInt = Field(description="How long the lease is valid, in ms")

class AcquireResponse(BaseModel):
    acquired: bool
    fencing_token: int | None = None
    expires_at: float | None = None  # unix epoch seconds

class RenewRequest(BaseModel):
    owner: str
    token: int
    ttl_ms: PositiveInt

class ReleaseRequest(BaseModel):
    owner: str
    token: int

# Demo: parse requests like a web framework would.
req = AcquireRequest.model_validate({"owner": "worker-A", "ttl_ms": 30_000})
print("parsed:", req)

ok  = AcquireResponse(acquired=True, fencing_token=17, expires_at=1_700_000_000.0)
no  = AcquireResponse(acquired=False)
print("ok :", ok.model_dump())
print("no :", no.model_dump())

# Validation catches garbage at the edge.
try:
    AcquireRequest.model_validate({"owner": "", "ttl_ms": -1})
except Exception as e:
    print("rejected bad input ✅ →", type(e).__name__)


## Why owner + token on `release`?

If you forget the owner check, a buggy or malicious client can release someone
else's lock. If you forget the token check, a *paused* owner that already lost
the lease could release the **next** owner's lock when it wakes up.

Both checks together make `release` **idempotent and safe**: if the caller no
longer holds the lock, the server returns `false` and does nothing.


In [ ]:
# Tiny in-memory implementation of the v2 API — no network yet, just the logic.
# We'll scale this up in Notebook 3 and attach a fenced resource.
import time, threading
from dataclasses import dataclass

@dataclass
class _Entry:
    owner: str
    token: int
    expires_at: float

class InMemoryLockService:
    def __init__(self):
        self._locks: dict[str, _Entry] = {}
        self._next_token = 0
        self._mu = threading.Lock()

    def _now(self): return time.time()

    def acquire(self, name, req: AcquireRequest) -> AcquireResponse:
        with self._mu:
            e = self._locks.get(name)
            if e and e.expires_at > self._now() and e.owner != req.owner:
                return AcquireResponse(acquired=False)
            self._next_token += 1
            e = _Entry(req.owner, self._next_token, self._now() + req.ttl_ms/1000)
            self._locks[name] = e
            return AcquireResponse(acquired=True, fencing_token=e.token, expires_at=e.expires_at)

    def release(self, name, req: ReleaseRequest) -> bool:
        with self._mu:
            e = self._locks.get(name)
            if e and e.owner == req.owner and e.token == req.token:
                del self._locks[name]
                return True
            return False

svc = InMemoryLockService()
r1 = svc.acquire("nightly-job", AcquireRequest(owner="A", ttl_ms=1000))
print("A acquires:", r1.model_dump())
r2 = svc.acquire("nightly-job", AcquireRequest(owner="B", ttl_ms=1000))
print("B denied  :", r2.model_dump())
# B tries to release A's lock with the wrong token → refused
print("B releases A's lock?", svc.release("nightly-job", ReleaseRequest(owner="B", token=999)))
print("A releases A's lock?", svc.release("nightly-job", ReleaseRequest(owner="A", token=r1.fencing_token)))
